# CMIP6-decadal CMCC monthly `tas` concat/subset error

When: 2026-09-22

This notebook preserves the reported request for ten CMCC-CM2-SR5 DCPP hindcast realizations initialized in 2008. It concatenates monthly near-surface air temperature (`tas`) along `realization`, then subsets the area `-104.0,26.0,-94.0,35.0` and all months in 2017–2019.

The reported service error is: `Process error: NoneType object is not callable.`

The requested time range is suspected to be incorrect for these hindcasts. This is a hypothesis; the available time coverage and the cause of the error have not yet been verified. The original request is preserved below for reproduction.


## Original workflow

Preserve the supplied collection order, dataset versions, and subset parameters. The escaped underscores in the pasted request are represented as ordinary underscores in the JSON keys and step references.


In [1]:
request = {
    "inputs": {
        "tas": [
            "c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r10i1p1f1.Amon.tas.gn.v20210719",
            "c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r1i1p1f1.Amon.tas.gn.v20210312",
            "c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r2i1p1f1.Amon.tas.gn.v20210312",
            "c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r3i1p1f1.Amon.tas.gn.v20210312",
            "c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r4i1p1f1.Amon.tas.gn.v20210312",
            "c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r5i1p1f1.Amon.tas.gn.v20210312",
            "c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r6i1p1f1.Amon.tas.gn.v20210312",
            "c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r7i1p1f1.Amon.tas.gn.v20210719",
            "c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r8i1p1f1.Amon.tas.gn.v20210719",
            "c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r9i1p1f1.Amon.tas.gn.v20210719"
        ]
    },
    "steps": {
        "concat_tas_1": {
            "run": "concat",
            "in": {
                "collection": "inputs/tas",
                "dims": "realization"
            }
        },
        "subset_tas_1": {
            "run": "subset",
            "in": {
                "collection": "concat_tas_1/output",
                "area": "-104.0,26.0,-94.0,35.0",
                "time_components": "month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|year:2017,2018,2019",
                "time": "2017/2019"
            }
        }
    },
    "outputs": {
        "output": "subset_tas_1/output"
    },
    "doc": "workflow"
}

request


{'inputs': {'tas': ['c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r10i1p1f1.Amon.tas.gn.v20210719',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r1i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r2i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r3i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r4i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r5i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r6i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r7i1p1f1.Amon.tas.gn.v20210719',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r8i1p1f1.Amon.tas.gn.v20210719',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r9i1p1f1.Amon.tas.gn.v202

## Build the equivalent Rooki workflow

Importing Rooki contacts the configured WPS service. Change `ROOK_URL` to test another deployment. The concat step has no time filter, matching the original request.


In [2]:
import json
import os

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

from rooki import operators as ops


In [3]:
tas = ops.Input("tas", request["inputs"]["tas"])
concat = ops.Concat(tas, dims=request["steps"]["concat_tas_1"]["in"]["dims"])
subset_parameters = request["steps"]["subset_tas_1"]["in"]
workflow = ops.Subset(
    concat,
    area=subset_parameters["area"],
    time_components=subset_parameters["time_components"],
    time=subset_parameters["time"],
)

serialized_request = json.loads(workflow._serialise())
assert serialized_request == request
serialized_request


{'inputs': {'tas': ['c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r10i1p1f1.Amon.tas.gn.v20210719',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r1i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r2i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r3i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r4i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r5i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r6i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r7i1p1f1.Amon.tas.gn.v20210719',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r8i1p1f1.Amon.tas.gn.v20210719',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r9i1p1f1.Amon.tas.gn.v202

## Reproduce and inspect the response

Set `RUN_REQUEST = True` to submit the workflow to the selected service. The cell displays the response and status so the actual service error can be recorded. Successful output URLs are listed without downloading data.


In [4]:
from time import perf_counter
from IPython.display import display

RUN_REQUEST = True

if RUN_REQUEST:
    started_at = perf_counter()
    response = workflow.orchestrate()
    print(f"Orchestration time: {perf_counter() - started_at:.1f} seconds")
    print(f"Succeeded: {response.ok}")
    print(f"Status: {response.status}")
    display(response)
    if response.ok:
        for url in response.download_urls():
            print(url)
else:
    print("Request not submitted; set RUN_REQUEST = True to reproduce it.")


Orchestration time: 9.6 seconds
Succeeded: False
Status: Process error: NoneType object is not callable


Process error: NoneType object is not callable

## Reported error and suspected cause

```text
Process error: NoneType object is not callable.
```

The request selects `2017/2019`, with `time_components` also selecting 2017, 2018, and 2019, from hindcasts initialized in 2008 (`s2008`). The requested years may extend beyond the available hindcast time coverage. The error message alone does not establish that this is the cause.

To investigate, first verify the available time coverage of all ten input collections. Then retry with a range within their shared coverage, updating both `time` and the year list in `time_components` consistently. Compare that response with the original request above before attributing the error to the time range.


## Test case: years within the decadal hindcast

Use 2009–2011, the first three full calendar years after the 2008 initialization, to avoid the end of the decadal forecast period. This case copies the original request and changes only `time` and the year list in `time_components`. All ten collections, their order and versions, the area, all months, and the concat step remain identical.

The actual availability of these years on the selected service is tested by the response below; a successful response would support, but not prove, the suspected time-range cause.


In [5]:
from copy import deepcopy

within_range_request = deepcopy(request)
within_range_parameters = within_range_request["steps"]["subset_tas_1"]["in"]
within_range_parameters["time"] = "2009/2011"
months = within_range_parameters["time_components"].split("|year:")[0]
within_range_parameters["time_components"] = f"{months}|year:2009,2010,2011"

within_range_tas = ops.Input("tas", within_range_request["inputs"]["tas"])
within_range_concat = ops.Concat(
    within_range_tas,
    dims=within_range_request["steps"]["concat_tas_1"]["in"]["dims"],
)
within_range_workflow = ops.Subset(
    within_range_concat,
    area=within_range_parameters["area"],
    time_components=within_range_parameters["time_components"],
    time=within_range_parameters["time"],
)

assert json.loads(within_range_workflow._serialise()) == within_range_request
assert request["steps"]["subset_tas_1"]["in"]["time"] == "2017/2019"
within_range_request


{'inputs': {'tas': ['c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r10i1p1f1.Amon.tas.gn.v20210719',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r1i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r2i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r3i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r4i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r5i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r6i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r7i1p1f1.Amon.tas.gn.v20210719',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r8i1p1f1.Amon.tas.gn.v20210719',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r9i1p1f1.Amon.tas.gn.v202

### Run the within-range request

This case uses the same `RUN_REQUEST` switch and service as the original reproduction. Inspect both responses to compare the outcome. Result data is not downloaded.


In [6]:
if RUN_REQUEST:
    started_at = perf_counter()
    within_range_response = within_range_workflow.orchestrate()
    print(f"Orchestration time: {perf_counter() - started_at:.1f} seconds")
    print(f"Succeeded: {within_range_response.ok}")
    print(f"Status: {within_range_response.status}")
    display(within_range_response)
    if within_range_response.ok:
        for url in within_range_response.download_urls():
            print(url)
else:
    print("Request not submitted; set RUN_REQUEST = True to test 2009–2011.")


Orchestration time: 12.6 seconds
Succeeded: True
Status: ProcessSucceeded


Metalink URL: http://rook7.cloud.dkrz.de:80/outputs/rook/9ed38d66-b681-11f1-bc07-fa163eb671ca/input.meta4, num files: 1

http://rook7.cloud.dkrz.de:80/outputs/rook/a582ad36-b681-11f1-970e-fa163eb671ca/tas_Amon_CMCC-CM2-SR5_dcppA-hindcast_r10i1p1f1_gn_20090114-20111216.nc


## Test case: partial overlap with the decadal hindcast

The 2009–2011 case above was reported to work as expected. This case requests 2007–2009: 2007 precedes the 2008 initialization, while 2009 overlaps the hindcast coverage demonstrated by that successful case. Any available months in 2008 are also selected.

Only `time` and the year list in `time_components` change. The collections, their order and versions, area, all months, and concat step remain identical. This tests whether the service returns the available intersection or rejects a partially overlapping request; this case was reported to fail as well.


In [7]:
from copy import deepcopy

partial_overlap_request = deepcopy(request)
partial_overlap_parameters = partial_overlap_request["steps"]["subset_tas_1"]["in"]
partial_overlap_parameters["time"] = "2007/2009"
months = partial_overlap_parameters["time_components"].split("|year:")[0]
partial_overlap_parameters["time_components"] = f"{months}|year:2007,2008,2009"

partial_overlap_tas = ops.Input("tas", partial_overlap_request["inputs"]["tas"])
partial_overlap_concat = ops.Concat(
    partial_overlap_tas,
    dims=partial_overlap_request["steps"]["concat_tas_1"]["in"]["dims"],
)
partial_overlap_workflow = ops.Subset(
    partial_overlap_concat,
    area=partial_overlap_parameters["area"],
    time_components=partial_overlap_parameters["time_components"],
    time=partial_overlap_parameters["time"],
)

assert json.loads(partial_overlap_workflow._serialise()) == partial_overlap_request
assert request["steps"]["subset_tas_1"]["in"]["time"] == "2017/2019"
partial_overlap_request


{'inputs': {'tas': ['c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r10i1p1f1.Amon.tas.gn.v20210719',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r1i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r2i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r3i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r4i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r5i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r6i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r7i1p1f1.Amon.tas.gn.v20210719',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r8i1p1f1.Amon.tas.gn.v20210719',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r9i1p1f1.Amon.tas.gn.v202

### Run the partial-overlap request

Use the same `RUN_REQUEST` switch and service as the preceding cases. Display the response and any successful output URLs without downloading result data.


In [8]:
if RUN_REQUEST:
    started_at = perf_counter()
    partial_overlap_response = partial_overlap_workflow.orchestrate()
    print(f"Orchestration time: {perf_counter() - started_at:.1f} seconds")
    print(f"Succeeded: {partial_overlap_response.ok}")
    print(f"Status: {partial_overlap_response.status}")
    display(partial_overlap_response)
    if partial_overlap_response.ok:
        for url in partial_overlap_response.download_urls():
            print(url)
else:
    print("Request not submitted; set RUN_REQUEST = True to test 2007–2009.")


Orchestration time: 6.4 seconds
Succeeded: False
Status: Process error: NoneType object is not callable


Process error: NoneType object is not callable

## Test case: partial overlap without `time_components`

The preceding 2007–2009 partial-overlap case was reported to fail. Repeat that request with `time_components` omitted entirely, retaining `time="2007/2009"`. All other request parameters remain identical. This isolates whether removing the component filter changes the outcome; this case was reported to fail as well.


In [9]:
from copy import deepcopy

time_only_request = deepcopy(partial_overlap_request)
time_only_parameters = time_only_request["steps"]["subset_tas_1"]["in"]
del time_only_parameters["time_components"]

time_only_tas = ops.Input("tas", time_only_request["inputs"]["tas"])
time_only_concat = ops.Concat(
    time_only_tas,
    dims=time_only_request["steps"]["concat_tas_1"]["in"]["dims"],
)
time_only_workflow = ops.Subset(
    time_only_concat,
    area=time_only_parameters["area"],
    time=time_only_parameters["time"],
)

time_only_serialized_request = json.loads(time_only_workflow._serialise())
assert time_only_serialized_request == time_only_request
assert "time_components" not in time_only_serialized_request["steps"]["subset_tas_1"]["in"]
assert "time_components" in partial_overlap_request["steps"]["subset_tas_1"]["in"]
time_only_serialized_request


{'inputs': {'tas': ['c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r10i1p1f1.Amon.tas.gn.v20210719',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r1i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r2i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r3i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r4i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r5i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r6i1p1f1.Amon.tas.gn.v20210312',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r7i1p1f1.Amon.tas.gn.v20210719',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r8i1p1f1.Amon.tas.gn.v20210719',
   'c3s-cmip6-decadal.DCPP.CMCC.CMCC-CM2-SR5.dcppA-hindcast.s2008-r9i1p1f1.Amon.tas.gn.v202

### Run the request without the component filter

Use the same `RUN_REQUEST` switch and service as the preceding cases. Display the response and any successful output URLs without downloading result data.


In [10]:
if RUN_REQUEST:
    started_at = perf_counter()
    time_only_response = time_only_workflow.orchestrate()
    print(f"Orchestration time: {perf_counter() - started_at:.1f} seconds")
    print(f"Succeeded: {time_only_response.ok}")
    print(f"Status: {time_only_response.status}")
    display(time_only_response)
    if time_only_response.ok:
        for url in time_only_response.download_urls():
            print(url)
else:
    print("Request not submitted; set RUN_REQUEST = True to test without time_components.")


Orchestration time: 6.5 seconds
Succeeded: False
Status: Process error: NoneType object is not callable


Process error: NoneType object is not callable

## Interpretation: requested coverage and failure message

The partial-overlap request also failed without `time_components`. We do not consider rejection of this request a subsetting error: decadal hindcasts have a finite forecast period, and the request asks for the full 2007–2009 range. Returning only the available year or portion of that range would give an incomplete result rather than what the user requested.

Rejecting a request whose full time range is unavailable is therefore reasonable. The issue is the diagnostic message: `Process error: NoneType object is not callable.` does not explain the coverage limitation. A useful response would state that the requested period exceeds the available hindcast coverage and identify the supported range. The reported test outcomes are consistent with this interpretation, but do not establish the internal cause of the exception.
